In [ ]:
from lib.dds import *
from lib.time import *
from lib.dataplot import *
import numpy as np

class Proportional:
    def __init__(self, k):
        self.k = k

    def evaluate(self, delta_t, u):
        return u * self.k

class Integrator:
    def __init__(self, ki):
        self.acc = 0
        self.ki = ki

    def evaluate(self, delta_t, u):
        out = self.acc + u * delta_t
        self.acc = out
        out *= self.ki
        return out

def saturate(inp, sat):
    if inp > sat:
        return (sat, True)
    elif inp < -sat:
        return (-sat, True)
    return (inp, False)

class PI:
    def __init__(self, k , ki, sat):
        self.P = Proportional(k)
        self.I = Integrator(ki)
        self.sat = sat
        self.in_sat = False

    def evaluate(self, delta_t, u):
        out = self.P.evaluate(delta_t, u)

        if self.in_sat:
            out += self.I.acc * self.I.ki

        else:
            out += self.I.evaluate(delta_t, u)

        out, self.in_sat = saturate(out, self.sat)

        return out

def inv_kin(v, r, l):
    k_rot = 0.6
    return  (1/r) * np.dot([[-1, 1, -k_rot * l], [1, 1, -k_rot * l], [1, -1, -k_rot * l], [-1, -1, -k_rot * l]], v)

class MecanumController:
    def __init__(self, k, ki, sat, R, L):
        self.PI_w1 = PI(k, ki, sat) 
        self.PI_w2 = PI(k, ki, sat)
        self.PI_w3 = PI(k, ki, sat)
        self.PI_w4 = PI(k, ki, sat)
        self.R = R
        self.L = L
        self.target = [0, 0, 0]

    def set_target(self, v):
        self.target = np.array(v)

    def evaluate(self, delta_t, u):
        v  = np.round(self.target - u, 3)
        err = inv_kin(v, self.R, self.L)
        err = np.round(err, 2)
        w1 = self.PI_w1.evaluate(delta_t, err[0])
        w2 = self.PI_w2.evaluate(delta_t, err[1])
        w3 = self.PI_w3.evaluate(delta_t, err[2])
        w4 = self.PI_w4.evaluate(delta_t, err[3])
        return [w1, w2, w3, w4]

class VirtualRobot:
    ACC = 0
    CRUISE = 1
    DEC = 2
    TARGET = 3
    
    def __init__(self,p_target,acc,v_max,dec):
        self.dir = 1 if p_target >= 0 else -1
        self.p_target = abs(p_target)
        self.acc = abs(acc)
        self.v_max = abs(v_max)
        self.dec = abs(dec)

        self.v = 0
        self.p = 0
        #self.t_dec = self.t_acc + (self.p_target/self.v_max) - (self.v_max /(2 * self.acc)) - (self.v_max)/(2 * self.dec)
        self.phase = VirtualRobot.ACC

    def evaluate(self,delta_t):
        match self.phase:
            case VirtualRobot.ACC:
                self.p = self.p + self.v * delta_t + ((1/2) * self.acc * delta_t * delta_t )
                self.v = self.v + self.acc * delta_t
                if self.v >= self.v_max:
                    self.phase = VirtualRobot.CRUISE
                if (self.p + (self.v * self.v) / ( 2 * self.dec)) >= self.p_target:
                    self.phase = VirtualRobot.DEC
            case VirtualRobot.CRUISE:
                self.p = self.p + self.v * delta_t
                if (self.p + (self.v * self.v) / ( 2 * self.dec)) >= self.p_target:
                    self.phase = VirtualRobot.DEC
            case VirtualRobot.DEC:
                self.p = self.p + self.v * delta_t - ((1/2) * self.dec * delta_t * delta_t)
                self.v = self.v - self.dec * delta_t
                if self.v < 0:
                    self.v = 0
                    self.phase = VirtualRobot.TARGET
            case VirtualRobot.TARGET:
                self.v = 0
        return self.v * self.dir

dpz = DataPlotter()
dpz.set_x("time (seconds)")
dpz.add_y("posZ", "posZ")
dpz.add_y("target", "target")
dpx = DataPlotter()
dpx.set_x("time (seconds)")
dpx.add_y("posX", "posX")
dpx.add_y("target", "target")
dpa = DataPlotter()
dpa.set_x("time (seconds)")
dpa.add_y("ang", "ang")
dpa.add_y("target", "target")

dp_vz = DataPlotter()
dp_vz.set_x("time (seconds)")
dp_vz.add_y("velZ", "velZ")
dp_vz.add_y("virtual","virtual")
dp_vx = DataPlotter()
dp_vx.set_x("time (seconds)")
dp_vx.add_y("velX", "velX")
dp_vx.add_y("virtual","virtual")
dp_va = DataPlotter()
dp_va.set_x("time (seconds)")
dp_va.add_y("velAng", "velAng")
dp_va.add_y("virtual","virtual")

dp_w1 = DataPlotter()
dp_w1.set_x("time (seconds)")
dp_w1.add_y("w1", "w1")
dp_w2 = DataPlotter()
dp_w2.set_x("time (seconds)")
dp_w2.add_y("w2", "w2")
dp_w3 = DataPlotter()
dp_w3.set_x("time (seconds)")
dp_w3.add_y("w3", "w3")
dp_w4 = DataPlotter()
dp_w4.set_x("time (seconds)")
dp_w4.add_y("w4", "w4")

dds = DDS()
dds.start()

dds.subscribe(["posZ", "posX", "ang", "velZ","velX","velAng"])

robot = MecanumController(2.5, 2, 50, 0.15, 1)
virtual_z = VirtualRobot(-15, 3, 5, 3)
virtual_x = VirtualRobot(-5, 3, 5, 3)
virtual_a = VirtualRobot(1, 0.2, 0.5, 0.2)

t = Time()
t.start()

while virtual_z.phase != VirtualRobot.TARGET or virtual_x.phase != VirtualRobot.TARGET or virtual_a.phase != VirtualRobot.TARGET:
    delta_t = t.elapsed()

    posZ = dds.wait("posZ")
    posX = dds.wait("posX")
    ang = dds.wait("ang")

    velZ = dds.wait("velZ")
    velX = dds.wait("velX")
    velAng = dds.wait("velAng")
    vector_v = np.array([velZ, velX, velAng])
    vz = virtual_z.evaluate(delta_t)
    vx = virtual_x.evaluate(delta_t)
    va = virtual_a.evaluate(delta_t)
    robot.set_target([vz, vx, va])
    w = robot.evaluate(delta_t, vector_v)

    dds.publish("w1", w[0], dds.DDS_TYPE_FLOAT)
    dds.publish("w2", w[1], dds.DDS_TYPE_FLOAT)
    dds.publish("w3", w[2], dds.DDS_TYPE_FLOAT)
    dds.publish("w4", w[3], dds.DDS_TYPE_FLOAT)

    dpz.append_x(t.get())
    dpz.append_y("posZ", posZ)
    dpz.append_y("target", -15)
    dpx.append_x(t.get())
    dpx.append_y("posX", posX)
    dpx.append_y("target", -5)
    dpa.append_x(t.get())
    dpa.append_y("ang", ang)
    dpa.append_y("target", np.rad2deg(1))

    dp_vz.append_x(t.get())
    dp_vz.append_y("velZ", velZ)
    dp_vz.append_y("virtual", vz)
    dp_vx.append_x(t.get())
    dp_vx.append_y("velX", velX)
    dp_vx.append_y("virtual", vx)
    dp_va.append_x(t.get())
    dp_va.append_y("velAng", velAng)
    dp_va.append_y("virtual", va)

    dp_w1.append_x(t.get())
    dp_w1.append_y("w1", w[0])
    dp_w2.append_x(t.get())
    dp_w2.append_y("w2", w[1])
    dp_w3.append_x(t.get())
    dp_w3.append_y("w3", w[2])
    dp_w4.append_x(t.get())
    dp_w4.append_y("w4", w[3])


dds.publish("w1", 0, dds.DDS_TYPE_FLOAT)
dds.publish("w2", 0, dds.DDS_TYPE_FLOAT)
dds.publish("w3", 0, dds.DDS_TYPE_FLOAT)
dds.publish("w4", 0, dds.DDS_TYPE_FLOAT)

dpz.plot()
dpx.plot()
dpa.plot()

dp_vz.plot()
dp_vx.plot()
dp_va.plot()

dp_w1.plot()
dp_w2.plot()
dp_w3.plot()
dp_w4.plot()

dds.stop()